In [ ]:
# ==============================================================
# DISCHARGE BIAS CORRECTION — RANDOM FOREST (RF)
# ==============================================================

import pandas as pd
import geopandas as gpd
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
import numpy as np
from sklearn.ensemble import RandomForestRegressor

# -------------------------- PATHS --------------------------
data_folder = Path(r"D:\Timeseries")
shapefile_path = r"D:\Discharge.shp"
coords_file = r"D:\Coordinates.csv"
station_seg_id_file = r"D:\Seg_ids.csv"
output_file = r"D:OUTPUT\CORRECTED_discharge.gpkg"

# -------------------------- FILE MAPPING --------------------------
file_mapping = {
    "A3 Naro Moru": "A3 Naro moru.csv",
    "A4 Naro Moru": "A4 Naro Moru.csv",
    "A6 Naro Moru": "A6 Naro Moru.csv",
    "A8 Burguret": "A8 Burguret.csv",
    "AB Ontulili": "AB Ontulili.csv"
}

# -------------------------- STEP 1: LOAD STATION DATA --------------------------
dfs = []
for station, filename in file_mapping.items():
    csv_file = data_folder / filename
    if csv_file.exists():
        df = pd.read_csv(csv_file)
        df['Station'] = station
        dfs.append(df)
    else:
        print(f"Warning: {csv_file} not found for station {station}.")
if not dfs:
    print("Error: No CSV files were loaded. Check file names and paths.")
    exit()
station_data = pd.concat(dfs, ignore_index=True)

# Ensure Date is in a consistent format (e.g., YYYY_MM) and extract month
station_data['Date'] = pd.to_datetime(station_data['Date'], format='%m/%d/%Y')
station_data['Month'] = station_data['Date'].dt.month

# -------------------------- STEP 2: TRAIN RANDOM FOREST MODELS --------------------------
correction_models = {}  # Nested dictionary: {station: {month: model}}
for station in file_mapping.keys():
    station_df = station_data[station_data['Station'] == station]
    if station_df.empty:
        print(f"Warning: No data found for station {station}")
        continue
    
    correction_models[station] = {}
    for month in range(1, 13):  # 1 to 12 for each month
        valid_data = station_df[(station_df['VegDischarge'] > 0.01) & 
                               (station_df['Obs'].notna()) & 
                               (station_df['VegDischarge'].notna()) & 
                               (station_df['Month'] == month)]
        if len(valid_data) > 1:  # Need at least 2 points for regression
            # Prepare data for Random Forest
            X = valid_data['VegDischarge'].values.reshape(-1, 1)  # Can add more features like Month or uparea
            y = valid_data['Obs'].values
            model = RandomForestRegressor(n_estimators=100, random_state=42)
            model.fit(X, y)
            correction_models[station][month] = model
            print(f"Random Forest model trained for {station}, month {month}: {len(valid_data)} data points")
        else:
            print(f"Warning: Insufficient data for Random Forest at {station}, month {month}")
            correction_models[station][month] = None  # Default to no correction
# -------------------------- STEP 3: LOAD SHAPEFILE --------------------------
coords = pd.read_csv(coords_file)
coords['Station'] = coords['Station'].str.strip()
shp_base = Path(shapefile_path).with_suffix('')
if not (shp_base.with_suffix('.shp').exists() and shp_base.with_suffix('.shx').exists() and shp_base.with_suffix('.dbf').exists()):
    print(f"Error: Missing shapefile components. Ensure {shp_base}.shp, {shp_base}.shx, and {shp_base}.dbf exist.")
    exit()
shp = gpd.read_file(shapefile_path)
print("Available columns in shapefile:", shp.columns.tolist())
print("Number of features loaded:", len(shp))
print("First few rows of shapefile data:\n", shp.head())

# Read the new CSV file for station-seg_id mapping
station_seg_id = pd.read_csv(data_folder / station_seg_id_file)
station_seg_id_map = dict(zip(station_seg_id['Station'], station_seg_id['seg_id']))

# Map stations to segments using seg_id
if 'seg_id' in shp.columns:
    shp['station'] = shp['seg_id'].map(
        {v: k for k, v in station_seg_id_map.items()}
    )
    print("Mapped stations:", shp['station'].dropna().unique())
else:
    print("Warning: 'seg_id' column not found in shapefile. Check column names.")
    exit()

# Extract monthly discharge columns (2000_01 to 2021_12)
discharge_cols = [col for col in shp.columns if col.startswith('20') and '_' in col]


# -------------------------- STEP 4: APPLY RF CORRECTION --------------------------
print("#Step 4: Apply monthly Random Forest correction to monthly discharges")
original_shp = shp.copy()  # Keep a copy of original data for plotting
for station, model_dict in correction_models.items():
    mask = shp['station'] == station
    if mask.any():
        print(f"Applying Random Forest correction to station {station}")
        for col in discharge_cols:
            month = pd.to_datetime(col, format='%Y_%m').month
            model = model_dict.get(month, None)
            if model is not None:
                # Apply Random Forest prediction
                X = shp.loc[mask, col].values.reshape(-1, 1)
                corrected_values = model.predict(X)
                # Check for negative values before clipping
                negative_count = (corrected_values < 0).sum()
                if negative_count > 0:
                    print(f"Warning: {negative_count} negative discharge values detected for {station}, {col} before clipping")
                # Clip negative values to zero
                shp.loc[mask, col] = np.clip(corrected_values, 0, None)
                print(f"Applied Random Forest correction to {col} for month {month}")
            else:
                print(f"No correction applied for {station}, {col} (month {month}) due to insufficient data")
    else:
        print(f"Warning: No segments matched for station {station}")


# -------------------------- STEP 5: VALIDATION --------------------------
print("#Step 5: Compute error metrics and plot time series with Random Forest correction")
metrics_summary = []
for station in file_mapping.keys():
    station_df = station_data[station_data['Station'] == station]
    if station_df.empty:
        print(f"Warning: No data found for station {station} in station_data")
        continue
    
    # Debug: Check if station exists in station_seg_id
    matching_rows = station_seg_id[station_seg_id['Station'] == station]
    if matching_rows.empty:
        print(f"Warning: No matching seg_id found for station {station} in {station_seg_id_file}. Check station names. Available stations: {station_seg_id['Station'].unique()}")
        continue
    
    # Use station_seg_id to map to the correct segment
    seg_id = matching_rows['seg_id'].iloc[0]
    station_seg_orig = original_shp[original_shp['seg_id'] == seg_id]
    station_seg_corr = shp[shp['seg_id'] == seg_id]
    if station_seg_orig.empty or station_seg_corr.empty:
        print(f"Warning: No segment found for seg_id {seg_id} in GeoPackage")
        continue
    
    # Extract time series
    discharge_cols = [col for col in station_seg_orig.columns if col.startswith('20') and '_' in col]
    print(f"Station {station}: GeoPackage discharge columns = {discharge_cols[:10]}... (total {len(discharge_cols)})")
    veg_discharge_orig = station_seg_orig[discharge_cols].values.flatten()
    veg_discharge_corr = station_seg_corr[discharge_cols].values.flatten()
    dates = pd.to_datetime(discharge_cols, format='%Y_%m', errors='coerce')
    valid_dates = dates[~dates.isna()]
    print(f"Station {station}: Parsed GeoPackage dates = {valid_dates[:5]}... (total {len(valid_dates)})")
    if len(valid_dates) == 0:
        print(f"Warning: No valid dates parsed from GeoPackage columns for {station}. Check column format.")
        continue
    
    # Use CSV dates for alignment
    csv_dates = station_df['Date'].dt.strftime('%Y_%m')
    geo_date_strs = dates.strftime('%Y_%m')
    csv_date_set = set(csv_dates)
    
    # Get indices where geo_date_strs are in csv_date_set
    mask = geo_date_strs.isin(csv_date_set)
    geo_indices = np.where(mask)[0]
    print(f"Station {station}: CSV Dates sample (str) = {csv_dates.head().tolist()}")
    print(f"Station {station}: GeoPackage Dates sample (str) = {geo_date_strs[:5].tolist()}")
    print(f"Station {station}: Unmatched CSV dates = {csv_date_set - set(geo_date_strs.dropna())}")
    print(f"Station {station}: Matched indices count = {len(geo_indices)}")
    
    # Extract overlapping data
    obs_series = station_df.set_index('Date')['Obs'].reindex(pd.to_datetime(csv_dates, format='%Y_%m')).loc[dates[geo_indices].date].values
    geo_dates = dates[geo_indices]
    veg_discharge_orig_valid = veg_discharge_orig[geo_indices]
    veg_discharge_corr_valid = veg_discharge_corr[geo_indices]
    
    # Create mask for valid data (excluding NaN and non-positive values)
    valid_mask = ~pd.isna(obs_series) & ~pd.isna(veg_discharge_orig_valid) & ~pd.isna(veg_discharge_corr_valid) & (obs_series > 0) & (veg_discharge_orig_valid > 0) & (veg_discharge_corr_valid > 0)
    
    # Debug: Check alignment and valid data count
    print(f"Station {station}: CSV Dates range = {station_df['Date'].min()} to {station_df['Date'].max()}")
    print(f"Station {station}: CSV Dates sample = {station_df['Date'].head().tolist()}")
    print(f"Station {station}: GeoPackage Dates range = {dates.min()} to {dates.max() if not dates.empty else 'N/A'}")
    print(f"Station {station}: Overlapping Geo Dates count = {len(geo_dates)}")
    print(f"Station {station}: Valid data points for plot = {np.sum(valid_mask)}")
    print(f"Station {station}: Reindexed Obs sample = {obs_series[:5].tolist()}")
    
    # Debug: Check data availability
    print(f"Station {station}: Obs length = {len(obs_series)}, non-NaN = {np.sum(~np.isnan(obs_series))}")
    print(f"Station {station}: Original VegDischarge length = {len(veg_discharge_orig_valid)}, non-NaN = {np.sum(~np.isnan(veg_discharge_orig_valid))}")
    print(f"Station {station}: Corrected VegDischarge length = {len(veg_discharge_corr_valid)}, non-NaN = {np.sum(~np.isnan(veg_discharge_corr_valid))}")
    
    # Remove NaN values for metric calculation
    metric_mask = ~pd.isna(obs_series) & ~pd.isna(veg_discharge_orig_valid) & ~pd.isna(veg_discharge_corr_valid)
    obs_valid = obs_series[metric_mask]
    veg_orig_valid = veg_discharge_orig_valid[metric_mask]
    veg_corr_valid = veg_discharge_corr_valid[metric_mask]
    
    if len(obs_valid) == 0 or len(veg_corr_valid) == 0:
        print(f"Warning: No valid data for validation at {station} after filtering.")
        continue
    
    # Calculate metrics between corrected values and observed
    nse = 1 - np.sum((obs_valid - veg_corr_valid) ** 2) / np.sum((obs_valid - np.mean(obs_valid)) ** 2)
    rmse = np.sqrt(mean_squared_error(obs_valid, veg_corr_valid))
    corr = np.corrcoef(obs_valid, veg_corr_valid)[0, 1]
    r2 = corr ** 2
    
    print(f"Validation metrics for {station} (seg_id {seg_id}):")
    print(f"  Nash-Sutcliffe Efficiency (NSE): {nse:.3f}")
    print(f"  Root Mean Square Error (RMSE): {rmse:.3f}")
    print(f"  R² (Coefficient of Determination): {r2:.3f}")
    print(f"  Correlation Coefficient: {corr:.3f}")
    
    # Store metrics for summary
    metrics_summary.append({
        'Station': station,
        'Seg_ID': seg_id,
        'NSE': nse,
        'RMSE': rmse,
        'R²': r2,
        'Correlation': corr
    })
    
    # Build continuous monthly date range from the earliest to latest GeoPackage date
    full_date_range = pd.date_range(start=dates.min(), end=dates.max(), freq='MS')

# Convert GeoPackage dates to a Series for alignment
    geo_series_index = pd.Series(geo_dates, index=geo_dates)

# Build continuous vectors for all 3 time series
# 1. Observed (from CSV)
    obs_full = pd.Series(np.nan, index=full_date_range)
    obs_aligned = pd.Series(obs_series, index=geo_dates)
    obs_full.update(obs_aligned)

# 2. Original VegDischarge
    veg_orig_full = pd.Series(np.nan, index=full_date_range)
    veg_orig_aligned = pd.Series(veg_discharge_orig_valid, index=geo_dates)
    veg_orig_full.update(veg_orig_aligned)

# 3. Corrected VegDischarge
    veg_corr_full = pd.Series(np.nan, index=full_date_range)
    veg_corr_aligned = pd.Series(veg_discharge_corr_valid, index=geo_dates)
    veg_corr_full.update(veg_corr_aligned)

# Plot with gaps automatically shown
    plt.figure(figsize=(12, 6))
    plt.plot(obs_full.index, obs_full.values,
             label='Observed', color='blue')

    plt.plot(veg_orig_full.index, veg_orig_full.values,
             label='Original VegDischarge', color='green', alpha=0.7)

    plt.plot(veg_corr_full.index, veg_corr_full.values,
             label='Corrected VegDischarge', color='red', alpha=0.7)

    plt.xlabel('Date', fontsize=16)
    plt.ylabel('Discharge', fontsize=16)
    plt.title(f'Time Series for {station} (seg_id {seg_id})', fontsize=20)
    plt.legend(fontsize=14)
    plt.grid(True)
    plt.xticks(rotation=45, fontsize=14)
    plt.yticks(fontsize=14)

    plt.tight_layout()
    plt.savefig(data_folder / f"{station.replace(' ', '_')}_plots_rf_corrected.png")
    plt.close()

# Save metrics summary to CSV
metrics_df = pd.DataFrame(metrics_summary)
metrics_df.to_csv(data_folder / "validation_metrics_rf_corrected.csv", index=False)
print(f"Validation metrics saved to {data_folder / 'validation_metrics_rf_corrected.csv'}")

# -------------------------- STEP 6: APPLY DISCHARGE-AREA-PROPORTION METHOD TO ESTIMATE MONTHLY DISCHARGES FOR OTHER SEGMENTS --------------------------

print("#Step 6: Apply Discharge-Area-Proportion method")
for station in file_mapping.keys():
    station_shp = shp[shp['station'] == station]
    if station_shp.empty:
        print(f"Warning: No data found for station {station} in shapefile")
        continue
    seg_id = station_shp['seg_id'].iloc[0]
    uparea_station = station_shp['uparea'].iloc[0] if 'uparea' in shp.columns else None
    
    if uparea_station is None:
        print(f"Warning: 'uparea' column not found or missing for station {station}")
        continue

    # Downstream estimation
    if 'NextDownID' not in shp.columns:
        print("Warning: 'NextDownID' column not found for downstream estimation")
    else:
        downstream_id = station_shp['NextDownID'].iloc[0]
        if pd.notna(downstream_id) and downstream_id in shp['seg_id'].values:
            downstream_shp = shp[shp['seg_id'] == downstream_id].copy()
            uparea_down = downstream_shp['uparea'].iloc[0]
            for col in discharge_cols:
                discharge = station_shp[col].iloc[0]
                estimated_discharge = discharge * (uparea_down / uparea_station)
                # Clip negative values to zero for downstream estimation
                if estimated_discharge < 0:
                    print(f"Warning: Negative discharge ({estimated_discharge:.3f}) for {station}, {col} in downstream estimation, clipping to 0")
                shp.loc[shp['seg_id'] == downstream_id, col] = max(estimated_discharge, 0)

    # Upstream estimation
    for up_col in ['up1', 'up2', 'up3', 'up4']:
        if up_col not in shp.columns:
            print(f"Warning: '{up_col}' column not found for upstream estimation")
        else:
            upstream_id = station_shp[up_col].iloc[0]
            if pd.notna(upstream_id) and upstream_id in shp['seg_id'].values:
                upstream_shp = shp[shp['seg_id'] == upstream_id].copy()
                uparea_up = upstream_shp['uparea'].iloc[0]
                for col in discharge_cols:
                    discharge = station_shp[col].iloc[0]
                    estimated_discharge = discharge * (uparea_up / uparea_station)
                    # Clip negative values to zero for upstream estimation
                    if estimated_discharge < 0:
                        print(f"Warning: Negative discharge ({estimated_discharge:.3f}) for {station}, {col} in upstream estimation, clipping to 0")
                    shp.loc[shp['seg_id'] == upstream_id, col] = max(estimated_discharge, 0)


# -------------------------- STEP 7: SAVE --------------------------

shp = shp.drop(columns=['station'])  # Remove the temporary station column
shp.to_file(output_file, driver="GPKG")
print(f"Updated GeoPackage saved to {output_file}")
